# Model comparison

Compare mean average precision (mAP) across `cell_dino`, `cell_profiler`, `dino`, `dynaclr`, and `subcell` models on four evaluation tasks: phenotypic activity, distinctiveness, manual complex consistency, and CORUM consistency.

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["svg.fonttype"] = "none"

FIGURES_DIR = Path("../../output/figure_2")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## CSV paths

Explicit path to every evaluation CSV used below.

**Before public release**, replace this cell with download instructions (or a pointer to the public dataset) and update the constants to match the released layout.

In [ ]:
# cell_dino
CELL_DINO_ACTIVITY          = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cell_dino/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_activity.csv"
CELL_DINO_DISTINCTIVENESS   = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cell_dino/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_distinctiveness.csv"
CELL_DINO_EBI               = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cell_dino/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_consistency_manual.csv"

# dino
DINO_ACTIVITY               = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/dino/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_activity.csv"
DINO_DISTINCTIVENESS        = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/dino/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_distinctiveness.csv"
DINO_EBI                    = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/dino/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_consistency_ebi.csv"

# cell_profiler
CELL_PROFILER_ACTIVITY        = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cellprofiler/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_activity.csv"
CELL_PROFILER_DISTINCTIVENESS = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cellprofiler/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_distinctiveness.csv"
CELL_PROFILER_EBI             = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cellprofiler/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_consistency_ebi.csv"

# dynaclr
DYNACLR_ACTIVITY            = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/dynaclr/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_activity.csv"
DYNACLR_DISTINCTIVENESS     = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/dynaclr/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_distinctiveness.csv"
DYNACLR_EBI                 = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/dynaclr/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_consistency_ebi.csv"

# subcell
SUBCELL_ACTIVITY            = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/subcell/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_activity.csv"
SUBCELL_DISTINCTIVENESS     = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/subcell/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_distinctiveness.csv"
SUBCELL_EBI                 = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/subcell/zscore_per_exp/paper_v1/all_livecell/fixed_80%/cosine/second_pca_consensus/metrics/phenotypic_consistency_ebi.csv"

## Load evaluation metrics

For each model, read the four evaluation CSVs and take the mean `mean_average_precision` per task.

In [ ]:
def summarise(activity_csv, distinctiveness_csv, ebi_csv):
    return {
        "mean_map_active":           pd.read_csv(activity_csv)["mean_average_precision"].mean(),
        "mean_map_distinct":         pd.read_csv(distinctiveness_csv)["mean_average_precision"].mean(),
        "mean_map_complexes_ebi": pd.read_csv(ebi_csv)["mean_average_precision"].mean(),
    }

records = {
    "cell_dino":     summarise(CELL_DINO_ACTIVITY,     CELL_DINO_DISTINCTIVENESS,     CELL_DINO_EBI),
    "dino":          summarise(DINO_ACTIVITY,          DINO_DISTINCTIVENESS,          DINO_EBI),
    "cell_profiler": summarise(CELL_PROFILER_ACTIVITY, CELL_PROFILER_DISTINCTIVENESS, CELL_PROFILER_EBI),
    "dynaclr":       summarise(DYNACLR_ACTIVITY,       DYNACLR_DISTINCTIVENESS,       DYNACLR_EBI),
    "subcell":       summarise(SUBCELL_ACTIVITY,       SUBCELL_DISTINCTIVENESS,       SUBCELL_EBI),
}

df = pd.DataFrame(records).T
df

## Mean average precision across all metric types

Single panel showing the mAP values across the four metric types for each model. Saves an SVG for the paper figure.

In [ ]:
MODELS = list(records.keys())
METRICS = ["mean_map_active", "mean_map_distinct", "mean_map_complexes_ebi"]
METRIC_LABELS = {
    "mean_map_active":           "Activity",
    "mean_map_distinct":         "Gene KO",
    "mean_map_complexes_ebi": "Protein Complex",
}

n_models = len(MODELS)
colors = plt.cm.tab10(np.linspace(0, 0.9, n_models))
bar_w = 0.8 / n_models
x = np.arange(len(METRICS))

fig, ax = plt.subplots(figsize=(6, 5))
for i, (model, color) in enumerate(zip(MODELS, colors)):
    offset = (i - (n_models - 1) / 2) * bar_w
    vals = [df.loc[model, m] for m in METRICS]
    ax.bar(x + offset, vals, width=bar_w * 0.9, label=model, color=color)

ax.set_xticks(x)
ax.set_xticklabels([METRIC_LABELS[m] for m in METRICS], fontsize=10)
ax.set_ylabel("mAP", fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=9, frameon=False, bbox_to_anchor=(0.75, 0.99), loc="upper left")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "model_comparison_map.svg", bbox_inches="tight")
plt.show()